# Pipeline to streamline the following tasks

 1. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, Duration, Channel]  

 2. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 3. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

##  Module Installation

In [1]:
%%capture
!pip install yt-dlp -U youtube-transcript-api pydub
!apt-get update
!apt-get install -y ffmpeg

## Module Imports

In [2]:
import yt_dlp
import pandas as pd
import youtube_transcript_api

## Main Variables

In [3]:
video_channel = []
video_IDs = []
video_durations = []
video_language = []
videos_LARGE = []
video_titles = []
video_language_INFO = {
    'Hindi'                                     : "hi",
    'Kashmiri'                                  : "ks",
    'Assamese'                                  : "as",
    "Marathi"                                   : "mr",
    "Gujarati"                                  : "gu",
    "Kannada"                                   : "kn",
    "Odia"                                      : "od",
    "Bengali"                                   : "bn"
}

### Constant for yt_dlp

In [4]:
ydl_opts = {
                'extract_flat': True,
                'skip_download': True,
                'ignoreerrors': True,
            }

In [5]:
#Helper Functions

def Larger_Video_CHECK(VIDEO_ID: str, DURATION: int):
    """Helper function to move larger videos to another area for processing later on"""
    if(DURATION > 1800):
        videos_LARGE.append(VIDEO_ID)
        return True

    return False


def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID



## Get YT Channel Usernames and Playlists

In [6]:
Sources = []

print("Please Provide the complete clean links")
def Get_Sources():
    while(1):
        print("Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):")
        a = input().strip()
        if(a in ['N','n']):
            break;

        if a=='': continue

        Sources.append(a)

Get_Sources()

Please Provide the complete clean links
Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):
https://www.youtube.com/playlist?list=PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb
Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):
n


### Function to fetch and extract video details to formulate the csv dataset

In [7]:
### Start Video ID, duration, extraction

def extract_video_details(Video_URL : str):

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(Video_URL, download=False)

        for entry in info['entries']:
            video_channel.append(info['channel'])
            video_IDs.append(entry['id'])
            video_durations.append(entry['duration'])
            video_titles.append(entry['title'])

In [8]:
%%capture
for source in Sources:
    extract_video_details(source)



In [9]:
print(len(video_channel) == len(video_durations) ==len(video_IDs))

True


### Form the .csv Dataset

In [10]:
##Form csv dataset
#.csv dataset creation with [SrNo. , YT_VIDEO_ID, Duration, CHANNEL]
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "YT_VIDEO_ID": video_IDs,
    "YT_VIDEO_TITLE": video_titles,
    "YT_VIDEO_LINK": [YT_Link_Generator(ID) for ID in video_IDs],
    "Duration": video_durations,
    "Channel": video_channel
})

df.to_csv("YT_DATASET.csv", index = False)

### DOWNLOAD AUDIO

In [12]:
# %%capture #Comment out if you dont want any outputs
def Don_Eladio(video_ID, video_Duration):
    flag = Larger_Video_CHECK(video_ID, video_Duration)

    if(flag): return

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': f'KrishiDarshan/{video_ID}/audio_{video_Duration}.%(ext)s',
        # 'http_headers': {
        #     'User-Agent':| 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        #     'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        #     'Accept-Language': 'en-us,en;q=0.5',
        #     'Sec-Fetch-Mode': 'navigate'
        # },
        # 'geo_bypass_country': 'US' # Or any other country code
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])


# l = len(video_IDs)
# for i in range(l):
#     Don_Eladio(video_IDs[i], video_durations[i])


l = len(video_IDs[:2])
for i in range(l):
    Don_Eladio(video_IDs[i], video_durations[i])

[youtube] Extracting URL: https://www.youtube.com/watch?v=SxgaGXcQ8ZY
[youtube] SxgaGXcQ8ZY: Downloading webpage


[youtube] SxgaGXcQ8ZY: Downloading android vr player API JSON


ERROR: [youtube] SxgaGXcQ8ZY: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


DownloadError: ERROR: [youtube] SxgaGXcQ8ZY: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

In [ ]:
# Zip audio files as a dataset

!zip -r KrishiDarshan.zip KrishiDarshan

In [ ]:
# %%capture
# for i in range(len(video_IDs[:1])):
#     flag = Larger_Video_CHECK(video_IDs[i], video_durations[i])

#     if(flag): continue

#     ydl_opts = {
#         'format': 'bestaudio/best',
#         'postprocessors': [{
#             'key': 'FFmpegExtractAudio',
#             'preferredcodec': 'mp3',
#             'preferredquality': '192',
#         }],
#         'outtmpl': f'KrishiDarshan/{video_IDs[i]}/audio.%(ext)s',
#     }

#     with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#         ydl.download([YT_Link_Generator(video_IDs[i])])

### Start Generating the transcript using whisper large v3 from Hugging Face

In [ ]:
# WHisper works by sliding a 30sec window to the audio, so there is no need to do it

In [11]:
# Fetch the audio files and start transcription
# Extract Zip file to datasets

#Currently Running on Colab

!unzip KrishiDarshan.zip

Archive:  KrishiDarshan.zip
   creating: content/
   creating: content/KrishiDarshan/
   creating: content/KrishiDarshan/BfMmAW-58ng/
  inflating: content/KrishiDarshan/BfMmAW-58ng/audio.mp3  
   creating: content/KrishiDarshan/ET8rwerJTg0/
  inflating: content/KrishiDarshan/ET8rwerJTg0/audio.mp3  


In [12]:
import os

root_dir = '/content/content/KrishiDarshan'
audio_file_paths = []
audio_durations = []

for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.mp3'):
            audio_file_paths.append(os.path.join(dirpath, filename))
            audio_durations.append(int(audio_file_paths[-1].split('_')[1].split('.')[0]))  ## must convert to integer

print(f"Found {len(audio_file_paths)} audio files:")
for path in audio_file_paths:
    print(path)


IndexError: list index out of range

In [ ]:
Transcripts = []
transcript = ""

In [13]:
%%capture
!pip install --upgrade transformers

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

# model = AutoModelForSpeechSeq2Seq.from_pretrained(
#     model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
# )
# model.to(device)

# processor = AutoProcessor.from_pretrained(model_id)

# pipe = pipeline(
#     "automatic-speech-recognition",
#     model=model,
#     tokenizer=processor.tokenizer,
#     feature_extractor=processor.feature_extractor,
#     dtype=torch_dtype,
#     device=device,
# )


In [ ]:
transcript = ""

for path in audio_file_paths:
    transcript = pipe(path, language='hi', return_timestamps=True)

    file = open(f'/content/KrishiDarshan/{path.split("/")[3]}/transcript.txt', 'w')
    file.write(transcript['text'])

    print(transcript['text'])

    file.close()

In [14]:
!pip install git+https://github.com/sanchit-gandhi/whisper-jax.git
!pip install --upgrade --no-deps --force-reinstall git+https://github.com/sanchit-gandhi/whisper-jax.git

  Cloning https://github.com/sanchit-gandhi/whisper-jax.git to /tmp/pip-req-build-_f2j_d5e
  Running command git clone --filter=blob:none --quiet https://github.com/sanchit-gandhi/whisper-jax.git /tmp/pip-req-build-_f2j_d5e
  Resolved https://github.com/sanchit-gandhi/whisper-jax.git to commit f983178a80ad37cf2f655777c26a74438b5d8690
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 29.2 MB/s eta 0:00:00
  Created wheel for whisper_jax: filename=whisper_jax-0.0.1-py3-none-any.whl size=60324 sha256=ba547e444614bdc72ce94ad6c0a8e58cbc956641b41de7c7886112c1f901b84b
  Stored in directory: /tmp/pip-ephem-wheel-cache-nl7

In [17]:
!ln -s /usr/local/lib/python3.12/dist-packages/whisper_jax /content/edit_target

In [18]:
from whisper_jax import FlaxWhisperForConditionalGeneration, FlaxWhisperPipline
import jax.numpy as jnp

pipeline = FlaxWhisperPipline('parthiv11/indic_whisper_nodcil', dtype=jnp.bfloat16)
transcript= pipeline('/content/content/KrishiDarshan/BfMmAW-58ng/audio.mp3')


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Some of the weights of FlaxWhisperForConditionalGeneration were initialized in float16 precision from the model checkpoint at parthiv11/indic_whisper_nodcil:
[('model', 'decoder', 'embed_positions', 'embedding'), ('model', 'decoder', 'embed_tokens', 'embedding'), ('model', 'decoder', 'layer_norm', 'bias'), ('model', 'decoder', 'layer_norm', 'scale'), ('model', 'decoder', 'layers', '0', 'encoder_attn', 'k_proj', 'kernel'), ('model', 'decoder', 'layers', '0', 'encoder_attn', 'out_proj', 'bias'), ('model', 'decoder', 'layers', '0', 'encoder_attn', 'out_proj', 'kernel'), ('model', 'decoder', 'layers', '0', 'encoder_attn', 'q_proj', 'bias'), ('model', 'decoder', 'layers', '0', 'encoder_attn', 'q_proj', 'kernel'), ('model', 'decoder', 'layers', '0', 'encoder_attn', 'v_proj', 'bias'), ('model', 'decoder', 'layers', '0', 'encoder_attn', 'v_proj', 'kernel'), ('model', 'decoder', 'layers', '0', 'encoder_attn_layer_norm', 'bias'), ('model', 'decoder', 'layers', '0', 'encoder_attn_layer_norm', 'sc

In [19]:
print(transcript['text'])

दिल्ली मुख्य समाचार नमस्कार कृषि दर्शन कार्यक्रम में आप सभी दर्शकों का स्वागत हैे रूप में कर रहे हैं और साथ में उसे बड़े चटकारे लेकर खा भी रहे हैं उसके बाद चने का उत्पादन के रूप में उपयोग होगा और उसके बाद प्रसंस्करण के रूप में भी इसकी मांग बढ़ जाती है तब हम किसानों का यह दायित्व और भी बढ़ जाता है कि इसके उत्पादन में हम कोई भी कोताही न बरतें इस समय चने में जो फलिलियाँ हैं उनकी फलियों में दाना स्वस्थ हो और हमें उत्पादन अच्छा मिले यही प्रयास हम किसानों का रहता है चना एक महत्वपूर्ण दलहनी फसल है भारत में कुल दलहनी उत्पादन का चना लगभग पैंतालीस प्रतिशत उत्पादन चने से आता है और कुल दलहनी फसलों के क्षेत्रफल का लगभग 35 से 40 प्रतिशत हिस्सा चने की खेती से होता है और इसी तरह इस तरह से आता�स बार फसल अच्छी है बारिश एक बार हुई है तो अब तक फसल में कोई ऐसा नुकसान नहीं हुआ है फली बनना शुरू हो गई है और इस अवस्था में किसानों को खास ध्यान रखना चाहिए क्योंकि अगर कोई कीट या बीमारी आती है तो इससे अत्यधिक नुकसान हो सकता है इस फली बनने के लिए �था में कुछ खास बातें हैं एक तो जैसे फली छेदक कीट नामक कीड़ा बहुत ही 

In [20]:
file = open("transcript.txt", 'w')
file.write(transcript['text'])

file.close()